In [8]:
from Bio.Align import PairwiseAligner

In [9]:
# 1. Pega aquí directamente tus secuencias como strings
secuencia_consenso = "MKLPDFIFGGATAAYQAEGATTDGKGVAWDYLENYWYTAEPASDFYRYPVDLLEFGVNGIRISIAWSRIFPGGEVNPKGVEYYHLFAECHKRHVEPFVTLHHFDTPELHGDFLNRNIFVYAFCFEFEVYWTTFNEIGPIGDGQYLVGKFPPGIYDKVFQSHHNMMVAHARAVKLFKDGYGEIGVVHALPTKYPDPNPDVRAAELEDIIHNKFILDATYLGYSETMEGVHILNGGLDEDFLAAKDLNDFLGINYYMSDWMAYDGEEIHNGTGKGSKYQKGVGRVPRTDWDWIYPQGLYDQIMRVKDYPNYKKIYITENGLGYKDEFDTVDDRIDYVKHLEVIDAIDGANVKGYFIWSLMDVFSWSNGYEKRYGLFYVDFETQERYPKKSAYWYKAETI"
secuencia_pdb = "MTKTLPKDFIFGGATAAYQAEGATHTDGKGPVAWDKYLEDNYWYTAEPASDFYHKYPVDLELAEEYGVNGIRISIAWSRIFPTGYGEVNEKGVEFYHKLFAECHKRHVEPFVTLHHFDTPEALHSNGDFLNRENIEHFIDYAAFCFEEFPEVNYWTTFNEIGPIGDGQYLVGKFPPGIKYDLAKVFQSHHNMMVSHARAVKLYKDKGYKGEIGVVHALPTKYPYDPENPADVRAAELEDIIHNKFILDATYLGHYSDKTMEGVNHILAENGGELDLRDEDFQALDAAKDLNDFLGINYYMSDWMQAFDGETEIIHNGKGEKGSSKYQIKGVGRRVAPDYVPRTDWDWIIYPEGLYDQIMRVKNDYPNYKKIYITENGLGYKDEFVDNTVYDDGRIDYVKQHLEVLSDAIADGANVKGYFIWSLMDVFSWSNGYEKRYGLFYVDFDTQERYPKKSAHWYKKLAETQVIE" 


In [10]:
# 2. Configuración del alineador Needleman-Wunsch
aligner = PairwiseAligner()
aligner.mode = 'global'
aligner.open_gap_score = -10
aligner.extend_gap_score = -0.5

In [11]:
# 3. Ejecutar alineamiento
alineamientos = aligner.align(secuencia_consenso, secuencia_pdb)
mejor_alineamiento = alineamientos[0]

print("Puntuación:", mejor_alineamiento.score)
# Esto imprimirá el alineamiento visual en consola para que lo revises
print(mejor_alineamiento)

Puntuación: 103.5
target            0 M---KLPDFIFGGATAAYQAEGAT----TDGKGVAWDYLENYWYTAEPASDFYRYPVDLL
                  0 |---...|||||||||||||||||----............|||||||||||||......|
query             0 MTKTLPKDFIFGGATAAYQAEGATHTDGKGPVAWDKYLEDNYWYTAEPASDFYHKYPVDL

target           53 ----EFGVNGIRISIAWSRIFP---GGEVNPKGVEYYHLFAECHKRHVEPFVTLHHFDTP
                 60 ----|.||||||||||||||||---|.........|..||||||||||||||||||||||
query            60 ELAEEYGVNGIRISIAWSRIFPTGYGEVNEKGVEFYHKLFAECHKRHVEPFVTLHHFDTP

target          106 E-----------LHGDFLNRNIFVYAFCFEFEVYWTTFNEIGPIGDGQYLVGKFPPGI--
                120 |-----------................|....|||||||||||||||||||||||||--
query           120 EALHSNGDFLNRENIEHFIDYAAFCFEEFPEVNYWTTFNEIGPIGDGQYLVGKFPPGIKY

target          153 -YDKVFQSHHNMMVAHARAVKLFKD--GYGEIGVVHALPTKY---PDPNPDVRAAELEDI
                180 -..|||||||||||.|||||||.||--..|||||||||||||---|....||||||||||
query           180 DLAKVFQSHHNMMVSHARAVKLYKDKGYKGEIGVVHALPTKYPYDPENPADVRAAELEDI

target

In [12]:
# 4. Mapeo de índices (Tu "traductor")
seq_cons_aln = mejor_alineamiento[0] 
seq_pdb_aln = mejor_alineamiento[1]  

mapa_coordenadas = {}
idx_consenso = 1 
idx_pdb = 1      

for aa_cons, aa_pdb in zip(seq_cons_aln, seq_pdb_aln):
    if aa_cons != '-' and aa_pdb != '-':
        mapa_coordenadas[idx_consenso] = idx_pdb
        idx_consenso += 1
        idx_pdb += 1
    elif aa_cons == '-':
        idx_pdb += 1
    elif aa_pdb == '-':
        mapa_coordenadas[idx_consenso] = None 
        idx_consenso += 1

In [13]:
# 5. Prueba de consulta
# Cambia el 180 por el número de posición que te arrojó InterPro/PROSITE
residuo_query = 370
print(f"\nEl residuo {residuo_query} del consenso es el {mapa_coordenadas.get(residuo_query)} en el PDB 1PBG.")


El residuo 370 del consenso es el 436 en el PDB 1PBG.


In [24]:
# 5. Prueba de consulta
# Cambia el 180 por el número de posición que te arrojó InterPro/PROSITE
residuo_query = 382
print(f"\nEl residuo {residuo_query} del consenso es el {mapa_coordenadas.get(residuo_query)} en el PDB 1PBG.")


El residuo 382 del consenso es el 448 en el PDB 1PBG.


In [51]:
list_of_locations, n_list_of_locations = [[245, 259], [312, 320], [346, 363], [370, 382]], len(list_of_locations)
names_entries = ['PRINTS_PR00131_1', 'PRINTS_PR00131_2', 'PRINTS_PR00131_3', 'PRINTS_PR00131_4']

for i in range(n_list_of_locations):
    list_actual = list_of_locations[i]
    n_lista_actual = len(list_actual) 
    for j in range(n_lista_actual):
        #print(f"\nEl residuo {list_actual[j]} del consenso es el {mapa_coordenadas.get(list_actual[j])} en el PDB 1PBG.")
        list_actual[j]= mapa_coordenadas.get(list_actual[j])
        list_of_locations[i] = list_actual
    #print('------ Fin del dominio ------')
    
print(list_of_locations)

[[290, 304], [371, 379], [412, 429], [436, 448]]


In [70]:
for name_entry, interval in zip(names_entries, list_of_locations):
    new_interval = '-'.join(str(i) for i in interval)
    cmd_select = f"cmd.select('{name_entry}', 'resi {new_interval}')"
    cmd_color = f"cmd.color('green', '{name_entry}')"
    print(cmd_select)
    print(cmd_color)
    print()

cmd.select('PRINTS_PR00131_1', 'resi 290-304')
cmd.color('green', 'PRINTS_PR00131_1')

cmd.select('PRINTS_PR00131_2', 'resi 371-379')
cmd.color('green', 'PRINTS_PR00131_2')

cmd.select('PRINTS_PR00131_3', 'resi 412-429')
cmd.color('green', 'PRINTS_PR00131_3')

cmd.select('PRINTS_PR00131_4', 'resi 436-448')
cmd.color('green', 'PRINTS_PR00131_4')



In [ ]:
cmd.select('inicio', 'resi 20-25')
